# LLM Maison — Distillation sur Google Colab (GPU gratuit)

**Avant de lancer :** pousse ton code sur GitHub (`git push origin main`), puis ouvre ce notebook dans Colab (Fichier → Ouvrir un notebook → onglet GitHub, ou uploade le .ipynb).

1. **Runtime → Changer le type d'exécution → GPU (T4)** puis exécuter les cellules dans l'ordre.
2. Les checkpoints sont sauvegardés dans **Google Drive** (`Mon Drive/llm_maison_checkpoints`).
3. Repo cloné : `https://github.com/j0hannj/MySpecialFriend.git` — modifie la cellule 2 si ton repo est ailleurs.

In [ ]:
# Vérifier que le GPU est activé
import subprocess
out = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(out.stdout or out.stderr or 'nvidia-smi non dispo')
assert 'Tesla T4' in (out.stdout or '') or 'GPU' in (out.stdout or ''), 'Active Runtime → Changer le type d\'exécution → GPU (T4)'

In [ ]:
# Cloner le repo (remplace l'URL par la tienne si besoin)
REPO_URL = 'https://github.com/j0hannj/MySpecialFriend.git'
REPO_DIR = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
import os
if not os.path.isdir(REPO_DIR):
    get_ipython().system(f'git clone --depth 1 {REPO_URL}')
else:
    get_ipython().system(f'cd {REPO_DIR} && git pull')
REPO_PATH = os.path.abspath(REPO_DIR)
os.chdir(REPO_PATH)
print('CWD:', os.getcwd())

In [ ]:
# Monter Google Drive et définir le dossier des checkpoints
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CKPT = '/content/drive/MyDrive/llm_maison_checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)
os.environ['CHECKPOINT_DIR'] = DRIVE_CKPT
print('Checkpoints sauvegardes dans:', DRIVE_CKPT)

In [ ]:
# Installer les dépendances
!pip install -q torch transformers accelerate bitsandbytes tqdm requests
!pip install -q -r requirements.txt 2>/dev/null || true

In [ ]:
# Lancer la distillation (3B, backend transformers, from scratch)
# Pour reprendre un checkpoint: enlève --fresh
import os
if 'REPO_PATH' in dir(): os.chdir(REPO_PATH)
!python -m training.distill --mode online --backend transformers --fresh